In [34]:
# imports

import os
import gradio as gr
import json

from openai import OpenAI
from dotenv import load_dotenv

In [35]:
# load environment variables
load_dotenv(override=True)

# get the API key from the environment variable
openai_api_key = os.getenv("OPENAI_API_KEY")

# initialize OpenAI client
client = OpenAI(api_key=openai_api_key)

In [36]:
MODEL = "gpt-5.6-luna"

In [37]:
SYSTEM_PROMPT = """
You are a helpful assistant for Airline called FlightAI.
Give short, courteous answers, not more than 1 sentence.
Always be accurate, If you don't know the answer say so, don't hallucinate.
"""

In [38]:
import sqlite3

DB_NAME = 'ticket_prices.db'

def get_connection():
    """Creates a connection with a busy timeout to avoid 'database is locked' errors."""
    return sqlite3.connect(DB_NAME, timeout=10)

def create_table():
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS prices (
                city TEXT PRIMARY KEY,
                price REAL
            )
        ''')
        conn.commit()

def insert_prices(ticket_data):
    """
    Inserts one or more employee rows.
    Skips duplicates based on the UNIQUE constraint on 'name'.
    Accepts a single tuple or a list of tuples.
    """
    if isinstance(ticket_data, tuple):
        ticket_data = [ticket_data]

    with get_connection() as conn:
        cursor = conn.cursor()

        cursor.execute("SELECT COUNT(*) FROM prices")
        before = cursor.fetchone()[0]

        cursor.executemany('''
            INSERT OR IGNORE INTO prices (city, price)
            VALUES (?, ?)
        ''', ticket_data)
        conn.commit()

        cursor.execute("SELECT COUNT(*) FROM prices")
        after = cursor.fetchone()[0]
        
        inserted = after - before
        skipped = len(ticket_data) - inserted
        
        print(f"Inserted {inserted} new row(s). Skipped {skipped} duplicate(s).")


# --- Usage ---
create_table()

insert_prices([  # bulk insert
    ('london', 8500),  # duplicate — skipped
    ('new york', 6200),
    ('berlin', 9100),
    ('singapore', 5800),
    ('dublin', 4500),
    ('madrid', 3769)
])

Inserted 0 new row(s). Skipped 6 duplicate(s).


In [39]:

def get_ticket_prices(destination_city):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        print(destination_city.lower())
        cursor.execute("SELECT price FROM prices WHERE city = ?", (destination_city.lower(),))
        result = cursor.fetchone()
    return f"The price of a ticket to {destination_city} is ${int(result[0])}" if result else f"No price data available for this {destination_city}"

In [40]:
import base64
from io import BytesIO
from PIL import Image

def artist(city):
    image_response = client.images.generate(
            model="gpt-image-1-mini",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))   

In [41]:
def talker(message):
    response = client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=message
    )
    return response.content

In [42]:
price_function = {
    "name": "get_ticket_prices",
    "description": "Get the price of a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to"
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [43]:
tools = [
    {
        "type": "function",
        "function": price_function
    }
]

In [44]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_prices":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            cities.append(city)
            price_details = get_ticket_prices(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

    return responses, cities    


In [45]:
def chat(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history
    response = client.chat.completions.create(messages=messages, model=MODEL, tools=tools, reasoning_effort="none")
    cities=[]
    image=None

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = client.chat.completions.create(messages=messages, model=MODEL, tools=tools, reasoning_effort="none")

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])

    return history, voice, image        


In [ ]:
# Callbacks (along with the chat() function above)
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


madrid
